In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from EinsumNetwork import Graph, EinsumNetwork
import datasets
import utils
import pickle
import math
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import statsmodels.stats.multitest as multi

In [ ]:
 ###                                         ###
 # This notebook assumes the models are loaded #
 ###                                         ###
### Element-wise statistic of the correlation matrices
sample_pool = pd.read_csv('/path/to/rds/hpc-work/scRNA/train.csv',header=0,index_col=0)
sample_pool=sample_pool[:round(0.8*len(sample_pool))]

#This function generates the gene-gene correlation matrices
def compute_heat(N_cell=sample_pool.shape[0]):
    #print(N_cell)
    ### Sample K cells with replacement from trainning set 
    base_sample=sample_pool.sample(n=N_cell, replace=True,axis=0)
    base_sample=base_sample.to_numpy()
    base_sample=np.where(base_sample>0,1,0)
    
    #####################
    # draw some samples #
    #####################
    
    samples_untrain = einet_0.sample(num_samples=N_cell).cpu().numpy()
    samples_train = einet.sample(num_samples=N_cell).cpu().numpy()
    #samples = samples.reshape((-1, 28, 28))
    
    ### Compute Pearson correlation
    
    simulation_corr=np.corrcoef(x=samples_train,rowvar=False)
    untrain_corr=np.corrcoef(x=samples_untrain,rowvar=False)
    base_corr=np.corrcoef(x=base_sample,rowvar=False)

    return [untrain_corr,simulation_corr,base_corr]

sample_list=[]
for i in range(100):
    print(i)
    samples_train = einet.sample(num_samples=5000).cpu().numpy()
    sample_list.append(samples_train)
Untrained_AEs=[]
Simulate_AEs=[]
Base_AEs=[]

for i in range(100):
    print(i)
    out1=compute_heat(5000)
    Untrained_AEs.append(out1[0])
    Simulate_AEs.append(out1[1])
    Base_AEs.append(out1[2])

In [ ]:
# Corr matrices from the trained model
stacked_Simu = np.stack(Simulate_AEs, axis=-1)
# Corr matrices from the bootstrapped datasets
stacked_Base = np.stack(Base_AEs, axis=-1)
# Corr matrices from the untrained model
stacked_untrained = np.stack(Untrained_AEs, axis=-1)

In [ ]:
### Welch's T-test: the benjamini-hochberg correction is performed when BH = True
def welch_tt(df1,df2,BH=False):
    mu1=np.nanmean(df1,axis=-1)
    v1=np.nanvar(df1,axis=-1)
    mu2=np.nanmean(df2,axis=-1)
    v2=np.nanvar(df2,axis=-1)
    t_score12 = (mu1-mu2)/np.sqrt(v1 + v2)
    N1=np.count_nonzero(~np.isnan(df1),axis=-1)
    N2=np.count_nonzero(~np.isnan(df2),axis=-1)
    dof12_wt=(v1+v2)**2/((v1**2/(N1-1)+v2**2/(N2-1)))
    p_value12_wt = stats.t.sf(np.abs(t_score12), dof12_wt) * 2
    if BH:
        kk=p_value12_wt.flatten()
        p_value12_wt = multi.multipletests(kk, method='fdr_bh')[1]
        p_value12_wt= p_value12_wt.reshape(1000,1000)
    return p_value12_wt

### Compute p-values and plot them on heatmap
### df1 should be stacked_Base, df2 should be either stacked_Simu or stacked_untrained
def plot_p(df1=None,df2=None,ttest=welch_tt,correct=False,plot_sig=False,row_ord=None,col_ord=None):
    p_value12=ttest(df1,df2,correct)
    '''if correct:
        kk=p_value12.flatten()
        p_value12 = multi.multipletests(kk, method='fdr_bh')[1]
        p_value12= p_value12.reshape(1000,1000)'''
    test_sig=np.where(p_value12<0.05,1,0)
    perc=round(100*np.sum(np.triu(test_sig))/((test_sig.shape[0]*(test_sig.shape[0]-1))/2),4)
    if plot_sig:
        dd=test_sig
    else:
        dd=p_value12
    dd=dd[row_order][:, col_order]
    cgt=sns.clustermap(dd,cmap="vlag",row_cluster=False,col_cluster=False,vmin=0,vmax=1)
    cgt.ax_row_dendrogram.set_visible(False) #suppress row dendrogram
    cgt.ax_col_dendrogram.set_visible(False)
    cgt.ax_heatmap.set_yticklabels('')
    cgt.ax_heatmap.set_xticklabels('')
    cgt.ax_heatmap.set_yticks([])
    cgt.ax_heatmap.set_xticks([])
    cgt.ax_heatmap.set_title("% of sig entries in the upper tri matrix {} %".format(perc))
    plt.show()
